# 🔤 Romanization Language Model
## Hindi & Punjabi ↔ Roman Script

This notebook provides a complete, self-contained romanization model that:
- Converts **Hindi (Devanagari)** and **Punjabi (Gurmukhi)** → Roman
- Converts **Romanized text** back → native script
- Learns new words via `train.csv` or manually
- Auto-detects the input script
- Uses fuzzy matching for unknown words

---
**No external libraries needed** — uses Python standard library only.

### Table of Contents
1. Character Maps (Hindi & Punjabi)
2. Word Dictionaries
3. Core Engine
4. RomanizationModel Class
5. Training from CSV
6. Demo & Examples
7. Interactive Translator
8. Vocabulary Inspector

---
## 1. Imports & Character Maps

In [23]:
import json
import os
import csv
from difflib import get_close_matches
from IPython.display import display, HTML

print("✅ Imports loaded successfully")

✅ Imports loaded successfully


In [24]:
# ══════════════════════════════════════════════════════
# DEVANAGARI (Hindi) CHARACTER MAPS
# ══════════════════════════════════════════════════════

HINDI_TO_ROMAN = {
    # Independent vowels
    'अ': 'a',  'आ': 'aa', 'इ': 'i',  'ई': 'ee', 'उ': 'u',  'ऊ': 'oo',
    'ए': 'e',  'ऐ': 'ai', 'ओ': 'o',  'औ': 'au', 'ऋ': 'ri', 'ॠ': 'ri',
    'ऑ': 'o',  'अं': 'an','अः': 'ah',

    # Vowel matras (dependent)
    'ा': 'aa', 'ि': 'i',  'ी': 'ee', 'ु': 'u',  'ू': 'oo',
    'े': 'e',  'ै': 'ai', 'ो': 'o',  'ौ': 'au', 'ृ': 'ri',
    'ं': 'n',  'ः': 'h',  'ँ': 'n',  '़': '',

    # Consonants
    'क': 'k',   'ख': 'kh',  'ग': 'g',   'घ': 'gh',  'ङ': 'ng',
    'च': 'ch',  'छ': 'chh', 'ज': 'j',   'झ': 'jh',  'ञ': 'ny',
    'ट': 't',   'ठ': 'th',  'ड': 'd',   'ढ': 'dh',  'ण': 'n',
    'त': 't',   'थ': 'th',  'द': 'd',   'ध': 'dh',  'न': 'n',
    'प': 'p',   'फ': 'ph',  'ब': 'b',   'भ': 'bh',  'म': 'm',
    'य': 'y',   'र': 'r',   'ल': 'l',   'व': 'v',   'ळ': 'l',
    'श': 'sh',  'ष': 'sh',  'स': 's',   'ह': 'h',
    'क्ष': 'ksh','त्र': 'tr','ज्ञ': 'gya',

    # Nukta consonants (Urdu-borrowed)
    'क़': 'q',  'ख़': 'kh', 'ग़': 'gh', 'ज़': 'z',
    'ड़': 'r',  'ढ़': 'rh', 'फ़': 'f',  'य़': 'y',

    # Halant (virama)
    '्': '',

    # Devanagari numerals
    '०':'0','१':'1','२':'2','३':'3','४':'4',
    '५':'5','६':'6','७':'7','८':'8','९':'9',

    # Punctuation
    '।': '.', '॥': '..', '\u200d': '', '\u200c': '',
}

ROMAN_TO_HINDI = {
    # Multi-char sequences first (greedy match)
    'ksh': 'क्ष', 'gya': 'ज्ञ', 'chh': 'छ',
    'kh': 'ख',   'gh': 'घ',   'ng': 'ङ',
    'ch': 'च',   'jh': 'झ',   'ny': 'ञ',
    'th': 'थ',   'dh': 'ध',   'ph': 'फ',
    'bh': 'भ',   'sh': 'श',   'tr': 'त्र',
    'aa': 'आ',   'ee': 'ई',   'oo': 'ऊ',
    'ai': 'ऐ',   'au': 'औ',   'ri': 'ऋ',
    'an': 'अं',  'ah': 'अः',
    # Single chars
    'a': 'अ', 'i': 'इ', 'u': 'उ', 'e': 'ए', 'o': 'ओ',
    'k': 'क', 'g': 'ग', 'j': 'ज', 'n': 'न',
    't': 'त', 'd': 'द', 'p': 'प', 'b': 'ब',
    'm': 'म', 'y': 'य', 'r': 'र', 'l': 'ल',
    'v': 'व', 'w': 'व', 's': 'स', 'h': 'ह',
    'f': 'फ़','z': 'ज़','q': 'क़',
}

print(f"✅ Hindi maps loaded — {len(HINDI_TO_ROMAN)} native chars, {len(ROMAN_TO_HINDI)} roman patterns")

✅ Hindi maps loaded — 89 native chars, 47 roman patterns


In [25]:
# ══════════════════════════════════════════════════════
# GURMUKHI (Punjabi) CHARACTER MAPS
# ══════════════════════════════════════════════════════

PUNJABI_TO_ROMAN = {
    # Independent vowels
    'ਅ': 'a',  'ਆ': 'aa', 'ਇ': 'i',  'ਈ': 'ee', 'ਉ': 'u', 'ਊ': 'oo',
    'ਏ': 'e',  'ਐ': 'ai', 'ਓ': 'o',  'ਔ': 'au',

    # Vowel matras
    'ਾ': 'aa', 'ਿ': 'i',  'ੀ': 'ee', 'ੁ': 'u',  'ੂ': 'oo',
    'ੇ': 'e',  'ੈ': 'ai', 'ੋ': 'o',  'ੌ': 'au',
    'ੰ': 'n',  'ਂ': 'n',  'ਃ': 'h',

    # Consonants
    'ਕ': 'k',   'ਖ': 'kh',  'ਗ': 'g',   'ਘ': 'gh',  'ਙ': 'ng',
    'ਚ': 'ch',  'ਛ': 'chh', 'ਜ': 'j',   'ਝ': 'jh',  'ਞ': 'ny',
    'ਟ': 't',   'ਠ': 'th',  'ਡ': 'd',   'ਢ': 'dh',  'ਣ': 'n',
    'ਤ': 't',   'ਥ': 'th',  'ਦ': 'd',   'ਧ': 'dh',  'ਨ': 'n',
    'ਪ': 'p',   'ਫ': 'ph',  'ਬ': 'b',   'ਭ': 'bh',  'ਮ': 'm',
    'ਯ': 'y',   'ਰ': 'r',   'ਲ': 'l',   'ਵ': 'v',   'ਲ਼': 'l',
    'ਸ਼': 'sh', 'ਸ': 's',   'ਹ': 'h',   'ਫ਼': 'f',  'ਜ਼': 'z',
    'ੜ': 'r',

    # Virama
    '੍': '',

    # Gurmukhi numerals
    '੦':'0','੧':'1','੨':'2','੩':'3','੪':'4',
    '੫':'5','੬':'6','੭':'7','੮':'8','੯':'9',

    # Punctuation
    '।': '.', '॥': '..'
}

ROMAN_TO_PUNJABI = {
    'chh': 'ਛ',  'ksh': 'ਕਸ਼', 'gya': 'ਗਿਆ',
    'kh': 'ਖ',  'gh': 'ਘ',   'ng': 'ਙ',
    'ch': 'ਚ',  'jh': 'ਝ',   'ny': 'ਞ',
    'th': 'ਥ',  'dh': 'ਧ',   'ph': 'ਫ',
    'bh': 'ਭ',  'sh': 'ਸ਼',
    'aa': 'ਆ',  'ee': 'ਈ',   'oo': 'ਊ',
    'ai': 'ਐ',  'au': 'ਔ',
    'a': 'ਅ', 'i': 'ਇ', 'u': 'ਉ', 'e': 'ਏ', 'o': 'ਓ',
    'k': 'ਕ', 'g': 'ਗ', 'j': 'ਜ', 'n': 'ਨ',
    't': 'ਤ', 'd': 'ਦ', 'p': 'ਪ', 'b': 'ਬ',
    'm': 'ਮ', 'y': 'ਯ', 'r': 'ਰ', 'l': 'ਲ',
    'v': 'ਵ', 'w': 'ਵ', 's': 'ਸ', 'h': 'ਹ',
    'f': 'ਫ਼','z': 'ਜ਼','q': 'ਕ',
}

print(f"✅ Punjabi maps loaded — {len(PUNJABI_TO_ROMAN)} native chars, {len(ROMAN_TO_PUNJABI)} roman patterns")

✅ Punjabi maps loaded — 71 native chars, 43 roman patterns


---
## 2. Word Dictionaries
These provide fast, accurate word-level lookups before falling back to character mapping.

In [26]:
# ══════════════════════════════════════════════════════
# HINDI WORD DICTIONARY  (native → roman)
# ══════════════════════════════════════════════════════

HINDI_WORD_DICT = {
    'नमस्ते': 'namaste',    'नमस्कार': 'namaskar',
    'धन्यवाद': 'dhanyavaad','शुक्रिया': 'shukriya',
    'हाँ': 'haan',          'नहीं': 'nahin',       'हां': 'haan',
    'मैं': 'main',          'तुम': 'tum',           'आप': 'aap',
    'वह': 'vah',            'यह': 'yah',            'हम': 'hum',
    'क्या': 'kya',          'कैसे': 'kaise',        'कहाँ': 'kahan',
    'कब': 'kab',            'क्यों': 'kyun',        'कौन': 'kaun',
    'खाना': 'khaana',       'पानी': 'paani',        'घर': 'ghar',
    'दोस्त': 'dost',        'परिवार': 'parivaar',   'प्यार': 'pyaar',
    'ठीक': 'theek',         'अच्छा': 'accha',       'बुरा': 'bura',
    'बड़ा': 'bada',          'छोटा': 'chhota',       'नया': 'naya',
    'पुराना': 'purana',     'काम': 'kaam',          'समय': 'samay',
    'आज': 'aaj',            'कल': 'kal',            'रात': 'raat',
    'दिन': 'din',           'सुबह': 'subah',        'शाम': 'shaam',
    'स्कूल': 'school',      'किताब': 'kitaab',      'पढ़ना': 'padhna',
    'लिखना': 'likhna',      'बोलना': 'bolna',       'सुनना': 'sunna',
    'देखना': 'dekhna',      'जाना': 'jaana',        'आना': 'aana',
    'करना': 'karna',        'होना': 'hona',         'रहना': 'rehna',
    'खुश': 'khush',         'दुखी': 'dukhi',        'भूखा': 'bhooka',
    'प्यासा': 'pyaasa',     'थका': 'thaka',         'सोना': 'sona',
}
HINDI_ROMAN_DICT = {v: k for k, v in HINDI_WORD_DICT.items()}

# ══════════════════════════════════════════════════════
# PUNJABI WORD DICTIONARY  (native → roman)
# ══════════════════════════════════════════════════════

PUNJABI_WORD_DICT = {
    'ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ': 'sat sri akal',
    'ਨਮਸਕਾਰ': 'namaskar',
    'ਧੰਨਵਾਦ': 'dhanvaad',
    'ਹਾਂ': 'haan',     'ਨਹੀਂ': 'nahin',
    'ਮੈਂ': 'main',     'ਤੁਸੀਂ': 'tuseen',   'ਅਸੀਂ': 'aseen',
    'ਉਹ': 'oh',        'ਇਹ': 'ih',
    'ਕੀ': 'ki',        'ਕਿਵੇਂ': 'kiven',    'ਕਿੱਥੇ': 'kitthe',
    'ਕਦੋਂ': 'kadon',   'ਕਿਉਂ': 'kiun',      'ਕੌਣ': 'kaun',
    'ਖਾਣਾ': 'khaana',  'ਪਾਣੀ': 'paani',     'ਘਰ': 'ghar',
    'ਦੋਸਤ': 'dost',    'ਪਰਿਵਾਰ': 'parivaar','ਪਿਆਰ': 'pyaar',
    'ਠੀਕ': 'theek',    'ਚੰਗਾ': 'changa',    'ਮਾੜਾ': 'maada',
    'ਵੱਡਾ': 'vadda',   'ਛੋਟਾ': 'chhota',    'ਨਵਾਂ': 'navan',
    'ਕੰਮ': 'kamm',     'ਸਮਾਂ': 'samaan',
    'ਅੱਜ': 'ajj',      'ਕੱਲ੍ਹ': 'kal',      'ਰਾਤ': 'raat',
    'ਦਿਨ': 'din',      'ਸਵੇਰ': 'saver',     'ਸ਼ਾਮ': 'shaam',
    'ਖੁਸ਼': 'khush',   'ਦੁਖੀ': 'dukhi',
    'ਪੜ੍ਹਨਾ': 'parhna','ਲਿਖਣਾ': 'likhna',
    'ਬੋਲਣਾ': 'bolna',  'ਸੁਣਨਾ': 'sunna',
    'ਦੇਖਣਾ': 'dekhna', 'ਜਾਣਾ': 'jaana',     'ਆਉਣਾ': 'aauna',
    'ਕਰਨਾ': 'karna',   'ਹੋਣਾ': 'hona',      'ਰਹਿਣਾ': 'rehna',
}
PUNJABI_ROMAN_DICT = {v: k for k, v in PUNJABI_WORD_DICT.items()}

print(f"✅ Hindi word dict: {len(HINDI_WORD_DICT)} entries")
print(f"✅ Punjabi word dict: {len(PUNJABI_WORD_DICT)} entries")

✅ Hindi word dict: 58 entries
✅ Punjabi word dict: 48 entries


---
## 3. Core Engine Functions

In [27]:
# ══════════════════════════════════════════════════════
# SCRIPT DETECTOR
# ══════════════════════════════════════════════════════

def detect_script(text: str) -> str:
    """Returns 'hindi', 'punjabi', 'roman', or 'mixed'."""
    devanagari = sum(1 for c in text if '\u0900' <= c <= '\u097F')
    gurmukhi   = sum(1 for c in text if '\u0A00' <= c <= '\u0A7F')
    latin      = sum(1 for c in text if c.isascii() and c.isalpha())
    total = devanagari + gurmukhi + latin or 1
    if devanagari / total > 0.4: return 'hindi'
    if gurmukhi   / total > 0.4: return 'punjabi'
    if latin       / total > 0.4: return 'roman'
    return 'mixed'


# ══════════════════════════════════════════════════════
# CHARACTER-LEVEL TRANSLITERATION
# ══════════════════════════════════════════════════════

def _char_map_romanize(text: str, char_map: dict) -> str:
    """Native script → Roman using greedy longest-match."""
    result, i = [], 0
    keys = sorted(char_map.keys(), key=len, reverse=True)
    while i < len(text):
        matched = False
        for key in keys:
            if text[i:i+len(key)] == key:
                result.append(char_map[key])
                i += len(key)
                matched = True
                break
        if not matched:
            result.append(text[i])
            i += 1
    return ''.join(result)


def _char_map_nativize(roman: str, char_map: dict) -> str:
    """Roman → native script using greedy longest-match."""
    result, i = [], 0
    roman = roman.lower()
    keys = sorted(char_map.keys(), key=len, reverse=True)
    while i < len(roman):
        matched = False
        for key in keys:
            if roman[i:i+len(key)] == key:
                result.append(char_map[key])
                i += len(key)
                matched = True
                break
        if not matched:
            result.append(roman[i])
            i += 1
    return ''.join(result)


def _fix_inherent_vowel(roman: str) -> str:
    """Insert inherent 'a' vowel between consecutive consonant clusters."""
    vowels = set('aeiou')
    out = []
    for i, ch in enumerate(roman):
        out.append(ch)
        if (ch.isalpha() and ch not in vowels and
                i + 1 < len(roman) and
                roman[i+1].isalpha() and roman[i+1] not in vowels):
            out.append('a')
    return ''.join(out)


print("✅ Core engine functions defined")

# Quick test
print(f"   detect_script('नमस्ते') = {detect_script('नमस्ते')}")
print(f"   detect_script('ਘਰ')    = {detect_script('ਘਰ')}")
print(f"   detect_script('hello') = {detect_script('hello')}")

✅ Core engine functions defined
   detect_script('नमस्ते') = hindi
   detect_script('ਘਰ')    = punjabi
   detect_script('hello') = roman


---
## 4. RomanizationModel Class
The main model class with persistent learning.

In [28]:
LEARNED_DB_PATH = 'learned_words.json'


def _load_learned() -> dict:
    if os.path.exists(LEARNED_DB_PATH):
        with open(LEARNED_DB_PATH, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {'hindi': {}, 'punjabi': {}}


def _save_learned(db: dict):
    with open(LEARNED_DB_PATH, 'w', encoding='utf-8') as f:
        json.dump(db, f, ensure_ascii=False, indent=2)


class RomanizationModel:
    """
    Bidirectional Hindi / Punjabi ↔ Roman transliteration model.

    3-Layer lookup strategy:
      1. Word-level dictionary  → fast exact match
      2. Fuzzy matching         → handles typos / close spellings
      3. Character-level map    → handles any unknown word

    All learned words persist to learned_words.json.
    """

    def __init__(self):
        self.learned = _load_learned()
        self._build_dicts()

    def _build_dicts(self):
        """Merge base dictionaries with learned words."""
        self.hindi_native   = {**HINDI_WORD_DICT,   **self.learned.get('hindi', {})}
        self.hindi_roman    = {v: k for k, v in self.hindi_native.items()}
        self.hindi_roman.update(HINDI_ROMAN_DICT)

        self.punjabi_native = {**PUNJABI_WORD_DICT,  **self.learned.get('punjabi', {})}
        self.punjabi_roman  = {v: k for k, v in self.punjabi_native.items()}
        self.punjabi_roman.update(PUNJABI_ROMAN_DICT)

    # ────────────────────────────────────────
    # Romanize: native → roman
    # ────────────────────────────────────────

    def romanize(self, text: str, lang: str = 'auto') -> str:
        """Convert native script → Roman. lang: 'hindi' | 'punjabi' | 'auto'"""
        if lang == 'auto':
            lang = detect_script(text)
            if lang == 'roman':  return text
            if lang == 'mixed':  lang = 'hindi'

        result = []
        for word in text.split():
            clean = word.strip('।॥ ,.!?')
            if lang == 'hindi' and clean in self.hindi_native:
                result.append(self.hindi_native[clean])
            elif lang == 'punjabi' and clean in self.punjabi_native:
                result.append(self.punjabi_native[clean])
            else:
                cmap = HINDI_TO_ROMAN if lang == 'hindi' else PUNJABI_TO_ROMAN
                roman = _char_map_romanize(clean, cmap)
                roman = _fix_inherent_vowel(roman)
                result.append(roman)
        return ' '.join(result)

    # ────────────────────────────────────────
    # De-romanize: roman → native
    # ────────────────────────────────────────

    def deromanize(self, roman_text: str, target_lang: str = 'hindi') -> str:
        """Convert Roman → native script. target_lang: 'hindi' | 'punjabi'"""
        result = []
        for word in roman_text.lower().split():
            clean = word.strip('.,!?')
            wdict = self.hindi_roman if target_lang == 'hindi' else self.punjabi_roman

            if clean in wdict:
                result.append(wdict[clean])               # exact
            else:
                close = get_close_matches(clean, list(wdict.keys()), n=1, cutoff=0.75)
                if close:
                    result.append(wdict[close[0]])         # fuzzy
                else:
                    cmap = ROMAN_TO_HINDI if target_lang == 'hindi' else ROMAN_TO_PUNJABI
                    result.append(_char_map_nativize(clean, cmap))  # char-level
        return ' '.join(result)

    # ────────────────────────────────────────
    # Process (auto-mode entry point)
    # ────────────────────────────────────────

    def process(self, text: str, mode: str = 'auto', target_lang: str = 'hindi') -> dict:
        """
        Main entry point.
        mode: 'auto' | 'romanize' | 'deromanize'
        Returns: dict with input, output, direction, detected_script
        """
        detected = detect_script(text)
        if mode == 'auto':
            if detected in ('hindi', 'punjabi'):
                output    = self.romanize(text, lang=detected)
                direction = f'{detected} → roman'
            else:
                output    = self.deromanize(text, target_lang=target_lang)
                direction = f'roman → {target_lang}'
        elif mode == 'romanize':
            lang      = detected if detected in ('hindi','punjabi') else target_lang
            output    = self.romanize(text, lang=lang)
            direction = f'{lang} → roman'
        else:
            output    = self.deromanize(text, target_lang=target_lang)
            direction = f'roman → {target_lang}'
        return {'input': text, 'output': output,
                'direction': direction, 'detected_script': detected}

    # ────────────────────────────────────────
    # Learning
    # ────────────────────────────────────────

    def learn(self, native_word: str, roman_word: str, lang: str) -> str:
        """Teach the model a new word pair and persist it."""
        lang = lang.lower().strip()
        if lang not in ('hindi', 'punjabi'):
            raise ValueError("lang must be 'hindi' or 'punjabi'")
        self.learned.setdefault(lang, {})[native_word.strip()] = roman_word.strip().lower()
        _save_learned(self.learned)
        self._build_dicts()
        return f"✅ Learned: {native_word} ↔ {roman_word} [{lang}]"

    def forget(self, native_word: str, lang: str) -> str:
        """Remove a learned word."""
        if lang in self.learned and native_word in self.learned[lang]:
            del self.learned[lang][native_word]
            _save_learned(self.learned)
            self._build_dicts()
            return f"✅ Removed: {native_word}"
        return f"⚠ Word not found in learned [{lang}]"

    def list_learned(self, lang: str = None) -> dict:
        return self.learned.get(lang, {}) if lang else self.learned

    def vocab_size(self) -> dict:
        return {
            'hindi_dict':    len(self.hindi_native),
            'punjabi_dict':  len(self.punjabi_native),
            'learned_hindi': len(self.learned.get('hindi',   {})),
            'learned_punjabi': len(self.learned.get('punjabi', {})),
        }


# ── Instantiate the model ──
model = RomanizationModel()
print("✅ RomanizationModel instantiated")
print(f"   Vocab: {model.vocab_size()}")

✅ RomanizationModel instantiated
   Vocab: {'hindi_dict': 84, 'punjabi_dict': 67, 'learned_hindi': 26, 'learned_punjabi': 19}


---
## 5. Training from CSV
Place your `train.csv` in the same folder as this notebook.

**CSV format:**
```
native_word,roman_word,language
नमस्ते,namaste,hindi
ਘਰ,ghar,punjabi
```

In [29]:
def train_from_csv(csv_path: str, model: RomanizationModel,
                   verbose: bool = True) -> dict:
    """
    Train the model from a CSV file.

    Required columns: native_word, roman_word, language
    language must be 'hindi' or 'punjabi'

    Returns stats dict: total, success, skipped, errors
    """
    if not os.path.exists(csv_path):
        print(f"❌ File not found: {csv_path}")
        return {'total': 0, 'success': 0, 'skipped': 0, 'errors': []}

    stats = {'total': 0, 'success': 0, 'skipped': 0, 'errors': []}

    with open(csv_path, 'r', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        # Normalize header names
        reader.fieldnames = [h.strip().lower() for h in reader.fieldnames]

        required = {'native_word', 'roman_word', 'language'}
        if not required.issubset(set(reader.fieldnames)):
            missing = required - set(reader.fieldnames)
            print(f"❌ CSV missing columns: {missing}")
            return stats

        for i, row in enumerate(reader, start=2):
            stats['total'] += 1
            try:
                native = (row.get('native_word') or '').strip()
                roman  = (row.get('roman_word')  or '').strip()
                lang   = (row.get('language')    or '').strip().lower()

                if not native or not roman or not lang:
                    stats['skipped'] += 1
                    if verbose: print(f"  ⚠ Row {i}: empty field — skipped")
                    continue

                if lang not in ('hindi', 'punjabi'):
                    stats['skipped'] += 1
                    if verbose: print(f"  ⚠ Row {i}: unknown lang '{lang}' — skipped")
                    continue

                msg = model.learn(native, roman, lang)
                stats['success'] += 1
                if verbose: print(f"  {msg}")

            except Exception as e:
                stats['errors'].append({'row': i, 'error': str(e)})
                if verbose: print(f"  ❌ Row {i}: {e}")

    return stats


def generate_sample_csv(path='train.csv'):
    """Generate a ready-to-use sample train.csv."""
    rows = [
        ('native_word','roman_word','language'),
        ('भाई','bhai','hindi'),('बहन','behen','hindi'),
        ('मम्मी','mammi','hindi'),('पापा','papa','hindi'),
        ('चाय','chai','hindi'),('रोटी','roti','hindi'),
        ('दाल','daal','hindi'),('सब्जी','sabzi','hindi'),
        ('मिठाई','mithai','hindi'),('बाजार','bazaar','hindi'),
        ('ज़िंदगी','zindagi','hindi'),('दुनिया','duniya','hindi'),
        ('देश','desh','hindi'),('गाँव','gaon','hindi'),
        ('शहर','shahar','hindi'),('नदी','nadi','hindi'),
        ('पहाड़','pahaad','hindi'),('जंगल','jangal','hindi'),
        ('बारिश','baarish','hindi'),('हवा','hawa','hindi'),
        ('ਭਰਾ','bhra','punjabi'),('ਭੈਣ','bhain','punjabi'),
        ('ਮਾਂ','maan','punjabi'),('ਪਿਤਾ','pita','punjabi'),
        ('ਚਾਹ','chah','punjabi'),('ਰੋਟੀ','roti','punjabi'),
        ('ਦਾਲ','daal','punjabi'),('ਸਬਜ਼ੀ','sabzi','punjabi'),
        ('ਬਾਜ਼ਾਰ','bazaar','punjabi'),('ਜ਼ਿੰਦਗੀ','zindagi','punjabi'),
        ('ਦੇਸ਼','desh','punjabi'),('ਪਿੰਡ','pind','punjabi'),
        ('ਸ਼ਹਿਰ','shahar','punjabi'),('ਦਰਿਆ','dariya','punjabi'),
        ('ਪਹਾੜ','pahaad','punjabi'),('ਜੰਗਲ','jangal','punjabi'),
    ]
    with open(path, 'w', encoding='utf-8', newline='') as f:
        csv.writer(f).writerows(rows)
    print(f"✅ Sample CSV written → {path}  ({len(rows)-1} word pairs)")


print("✅ Training functions defined")

✅ Training functions defined


In [30]:
# ── Generate sample train.csv if it doesn't exist ──
if not os.path.exists('train.csv'):
    generate_sample_csv('train.csv')
else:
    print("✅ train.csv already exists — using it as-is")

✅ train.csv already exists — using it as-is


In [31]:
# ══════════════════════════════════════════════════════
# TRAIN THE MODEL FROM train.csv
# Change verbose=False to suppress per-row output
# ══════════════════════════════════════════════════════

print("Training from train.csv...\n")
stats = train_from_csv('train.csv', model=model, verbose=True)

print(f"\n{'─'*45}")
print(f"  Total rows  : {stats['total']}")
print(f"  ✅ Learned   : {stats['success']}")
print(f"  ⚠  Skipped  : {stats['skipped']}")
print(f"  ❌ Errors   : {len(stats['errors'])}")
print(f"\n  Vocabulary after training:")
for k, v in model.vocab_size().items():
    print(f"    {k:<22}: {v}")
print(f"{'─'*45}")

Training from train.csv...

  ✅ Learned: भाई ↔ bhai [hindi]
  ✅ Learned: बहन ↔ behen [hindi]
  ✅ Learned: मम्मी ↔ mammi [hindi]
  ✅ Learned: पापा ↔ papa [hindi]
  ✅ Learned: चाय ↔ chai [hindi]
  ✅ Learned: रोटी ↔ roti [hindi]
  ✅ Learned: दाल ↔ daal [hindi]
  ✅ Learned: सब्जी ↔ sabzi [hindi]
  ✅ Learned: मिठाई ↔ mithai [hindi]
  ✅ Learned: बाजार ↔ bazaar [hindi]
  ✅ Learned: ज़िंदगी ↔ zindagi [hindi]
  ✅ Learned: दुनिया ↔ duniya [hindi]
  ✅ Learned: देश ↔ desh [hindi]
  ✅ Learned: गाँव ↔ gaon [hindi]
  ✅ Learned: शहर ↔ shahar [hindi]
  ✅ Learned: नदी ↔ nadi [hindi]
  ✅ Learned: पहाड़ ↔ pahaad [hindi]
  ✅ Learned: जंगल ↔ jangal [hindi]
  ✅ Learned: बारिश ↔ baarish [hindi]
  ✅ Learned: हवा ↔ hawa [hindi]
  ✅ Learned: ਭਰਾ ↔ bhra [punjabi]
  ✅ Learned: ਭੈਣ ↔ bhain [punjabi]
  ✅ Learned: ਮਾਂ ↔ maan [punjabi]
  ✅ Learned: ਪਿਤਾ ↔ pita [punjabi]
  ✅ Learned: ਚਾਹ ↔ chah [punjabi]
  ✅ Learned: ਰੋਟੀ ↔ roti [punjabi]
  ✅ Learned: ਦਾਲ ↔ daal [punjabi]
  ✅ Learned: ਸਬਜ਼ੀ ↔ sabzi [punjabi]
  ✅ Learne

---
## 6. Demo & Examples

In [32]:
# ══════════════════════════════════════════════════════
# HINDI → ROMAN
# ══════════════════════════════════════════════════════

hindi_tests = [
    'नमस्ते',
    'धन्यवाद',
    'मैं ठीक हूँ',
    'क्या हाल है',
    'खाना खाओ',
    'आज मौसम अच्छा है',
    'मेरा घर यहाँ है',
    'दिल्ली भारत की राजधानी है',
]

print("🔤 Hindi → Roman")
print("─" * 50)
for text in hindi_tests:
    result = model.romanize(text, lang='hindi')
    print(f"  {text:<30}→  {result}")

🔤 Hindi → Roman
──────────────────────────────────────────────────
  नमस्ते                        →  namaste
  धन्यवाद                       →  dhanyavaad
  मैं ठीक हूँ                   →  main theek hoon
  क्या हाल है                   →  kya haal hai
  खाना खाओ                      →  khaana kahaao
  आज मौसम अच्छा है              →  aaj mausam accha hai
  मेरा घर यहाँ है               →  meraa ghar yahaan hai
  दिल्ली भारत की राजधानी है     →  dilalee bahaarat kee raajadahaanee hai


In [33]:
# ══════════════════════════════════════════════════════
# PUNJABI → ROMAN
# ══════════════════════════════════════════════════════

punjabi_tests = [
    'ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ',
    'ਧੰਨਵਾਦ',
    'ਘਰ',
    'ਪਾਣੀ',
    'ਖਾਣਾ',
    'ਤੁਸੀਂ ਕਿਵੇਂ ਹੋ',
    'ਮੈਂ ਠੀਕ ਹਾਂ',
]

print("🔤 Punjabi → Roman")
print("─" * 50)
for text in punjabi_tests:
    result = model.romanize(text, lang='punjabi')
    print(f"  {text:<25}→  {result}")

🔤 Punjabi → Roman
──────────────────────────────────────────────────
  ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ            →  sati saree akaal
  ਧੰਨਵਾਦ                   →  dhanvaad
  ਘਰ                       →  ghar
  ਪਾਣੀ                     →  paani
  ਖਾਣਾ                     →  khaana
  ਤੁਸੀਂ ਕਿਵੇਂ ਹੋ           →  tuseen kiven ho
  ਮੈਂ ਠੀਕ ਹਾਂ              →  main theek haan


In [34]:
# ══════════════════════════════════════════════════════
# ROMAN → HINDI  (de-romanization)
# ══════════════════════════════════════════════════════

roman_hindi_tests = [
    'namaste',
    'dhanyavaad',
    'kya haal hai',
    'mera naam kya hai',
    'khana khao',
    'ghar kahan hai',
    'aaj subah',
]

print("🔄 Roman → Hindi (Devanagari)")
print("─" * 50)
for text in roman_hindi_tests:
    result = model.deromanize(text, target_lang='hindi')
    print(f"  {text:<25}→  {result}")

🔄 Roman → Hindi (Devanagari)
──────────────────────────────────────────────────
  namaste                  →  नमस्ते
  dhanyavaad               →  धन्यवाद
  kya haal hai             →  क्या हवा चाय
  mera naam kya hai        →  मएरअ नया क्या चाय
  khana khao               →  खाना खअओ
  ghar kahan hai           →  घर कहाँ चाय
  aaj subah                →  आज सुबह


In [35]:
# ══════════════════════════════════════════════════════
# ROMAN → PUNJABI  (de-romanization)
# ══════════════════════════════════════════════════════

roman_punjabi_tests = [
    'sat sri akal',
    'dhanvaad',
    'ghar',
    'paani',
    'khaana',
    'tuseen kiven ho',
]

print("🔄 Roman → Punjabi (Gurmukhi)")
print("─" * 50)
for text in roman_punjabi_tests:
    result = model.deromanize(text, target_lang='punjabi')
    print(f"  {text:<25}→  {result}")

🔄 Roman → Punjabi (Gurmukhi)
──────────────────────────────────────────────────
  sat sri akal             →  ਸਅਤ ਸਰਇ ਕੱਲ੍ਹ
  dhanvaad                 →  ਧੰਨਵਾਦ
  ghar                     →  ਘਰ
  paani                    →  ਪਾਣੀ
  khaana                   →  ਖਾਣਾ
  tuseen kiven ho          →  ਤੁਸੀਂ ਕਿਵੇਂ ਹਓ


In [36]:
# ══════════════════════════════════════════════════════
# AUTO-DETECT MODE
# ══════════════════════════════════════════════════════

auto_tests = [
    'नमस्ते कैसे हो',       # Hindi → auto romanize
    'ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ',       # Punjabi → auto romanize
    'namaste kaise ho',      # Roman → auto deromanize (to hindi)
    'dhanyavaad',            # Roman → auto deromanize
]

print("🤖 Auto-Detect Mode (roman → hindi by default)")
print("─" * 60)
for text in auto_tests:
    r = model.process(text)
    print(f"  Input    : {r['input']}")
    print(f"  Detected : {r['detected_script']}")
    print(f"  Output   : {r['output']}")
    print(f"  Direction: {r['direction']}")
    print()

🤖 Auto-Detect Mode (roman → hindi by default)
────────────────────────────────────────────────────────────
  Input    : नमस्ते कैसे हो
  Detected : hindi
  Output   : namaste kaise ho
  Direction: hindi → roman

  Input    : ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ
  Detected : punjabi
  Output   : sati saree akaal
  Direction: punjabi → roman

  Input    : namaste kaise ho
  Detected : roman
  Output   : नमस्ते कैसे हओ
  Direction: roman → hindi

  Input    : dhanyavaad
  Detected : roman
  Output   : धन्यवाद
  Direction: roman → hindi



---
## 7. Learning New Words

In [37]:
# ══════════════════════════════════════════════════════
# TEACH THE MODEL NEW WORDS
# model.learn(native_word, roman_word, language)
# ══════════════════════════════════════════════════════

# ── Add new Hindi words ──
print(model.learn('रिक्शा',   'riksha',   'hindi'))
print(model.learn('समोसा',    'samosa',   'hindi'))
print(model.learn('जुगाड़',   'jugaad',   'hindi'))
print(model.learn('बॉलीवुड', 'bollywood', 'hindi'))

# ── Add new Punjabi words ──
print(model.learn('ਪੰਜਾਬ',    'punjab',   'punjabi'))
print(model.learn('ਲੰਗਰ',     'langar',   'punjabi'))
print(model.learn('ਭੰਗੜਾ',   'bhangra',  'punjabi'))

# ── Verify ──
print()
print("Verification:")
print(f"  रिक्शा → {model.romanize('रिक्शा', 'hindi')}")
print(f"  riksha → {model.deromanize('riksha', 'hindi')}")
print(f"  ਲੰਗਰ → {model.romanize('ਲੰਗਰ', 'punjabi')}")
print(f"  bhangra → {model.deromanize('bhangra', 'punjabi')}")

✅ Learned: रिक्शा ↔ riksha [hindi]
✅ Learned: समोसा ↔ samosa [hindi]
✅ Learned: जुगाड़ ↔ jugaad [hindi]
✅ Learned: बॉलीवुड ↔ bollywood [hindi]
✅ Learned: ਪੰਜਾਬ ↔ punjab [punjabi]
✅ Learned: ਲੰਗਰ ↔ langar [punjabi]
✅ Learned: ਭੰਗੜਾ ↔ bhangra [punjabi]

Verification:
  रिक्शा → riksha
  riksha → रिक्शा
  ਲੰਗਰ → langar
  bhangra → ਭੰਗੜਾ


In [38]:
# ══════════════════════════════════════════════════════
# ADD YOUR OWN WORDS HERE
# ══════════════════════════════════════════════════════

# ✏️ Edit these to add your custom vocabulary:

custom_words = [
    # (native_word,   roman_word,   language)
    ('मेरा नाम',     'mera naam',  'hindi'),
    ('आपका स्वागत', 'aapka swagat', 'hindi'),
    # Add more rows here...
]

for native, roman, lang in custom_words:
    print(model.learn(native, roman, lang))

✅ Learned: मेरा नाम ↔ mera naam [hindi]
✅ Learned: आपका स्वागत ↔ aapka swagat [hindi]


---
## 8. Interactive Translator
A simple widget-style translator you can run in-notebook.

In [39]:
# ══════════════════════════════════════════════════════
# INTERACTIVE TRANSLATOR  (no external widgets needed)
# Just edit the variables below and re-run this cell
# ══════════════════════════════════════════════════════

# ✏️ Edit these:
INPUT_TEXT  = "नमस्ते कैसे हो भाई"   # ← your text here
MODE        = 'auto'                   # 'auto' | 'romanize' | 'deromanize'
TARGET_LANG = 'hindi'                  # used when mode='deromanize': 'hindi' | 'punjabi'

# ── Run ──
result = model.process(INPUT_TEXT, mode=MODE, target_lang=TARGET_LANG)

# ── Display ──
display(HTML(f"""
<div style="font-family:monospace; background:#f8f8f8; padding:16px;
            border-left:4px solid #4a90d9; border-radius:4px; margin:8px 0">
  <div style="color:#666; font-size:12px;">DIRECTION</div>
  <div style="font-size:16px; font-weight:bold; margin:4px 0 12px">{result['direction']}</div>

  <div style="display:grid; grid-template-columns:1fr 1fr; gap:16px">
    <div>
      <div style="color:#666; font-size:12px;">INPUT  ({result['detected_script']})</div>
      <div style="font-size:20px; background:white; padding:10px; border-radius:4px;
                  border:1px solid #ddd; margin-top:4px">{result['input']}</div>
    </div>
    <div>
      <div style="color:#666; font-size:12px;">OUTPUT</div>
      <div style="font-size:20px; background:white; padding:10px; border-radius:4px;
                  border:1px solid #4a90d9; margin-top:4px; color:#2c5f9e">{result['output']}</div>
    </div>
  </div>
</div>
"""))

In [40]:
# ══════════════════════════════════════════════════════
# BATCH TRANSLATION
# Translate a list of sentences at once
# ══════════════════════════════════════════════════════

sentences = [
    'नमस्ते दोस्त',
    'ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ',
    'mera ghar kahan hai',
    'khana bahut accha hai',
    'ਪਾਣੀ ਦਿਓ',
    'aaj bahut garmi hai',
]

print("Batch Translation Results")
print("═" * 60)
for s in sentences:
    r = model.process(s)
    print(f"  [{r['detected_script']:<8}] {s:<30} → {r['output']}")

Batch Translation Results
════════════════════════════════════════════════════════════
  [hindi   ] नमस्ते दोस्त                   → namaste dost
  [punjabi ] ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ                  → sati saree akaal
  [roman   ] mera ghar kahan hai            → मएरअ घर कहाँ चाय
  [roman   ] khana bahut accha hai          → खाना बअःउत अच्छा चाय
  [punjabi ] ਪਾਣੀ ਦਿਓ                       → paani dio
  [roman   ] aaj bahut garmi hai            → आज बअःउत गअरमइ चाय


---
## 9. Vocabulary Inspector

In [41]:
# ══════════════════════════════════════════════════════
# VOCABULARY STATISTICS
# ══════════════════════════════════════════════════════

vs = model.vocab_size()
total = sum(vs.values())

display(HTML(f"""
<div style="font-family:monospace; padding:12px">
  <h3 style="margin:0 0 12px">📚 Vocabulary Statistics</h3>
  <table style="border-collapse:collapse; width:400px">
    {''.join(
        f'<tr><td style="padding:6px 16px 6px 0; color:#555">{k}</td>'
        f'<td style="padding:6px; font-weight:bold; color:#2c5f9e">{v}</td></tr>'
        for k, v in vs.items()
    )}
    <tr style="border-top:1px solid #ccc">
      <td style="padding:8px 16px 6px 0; font-weight:bold">TOTAL</td>
      <td style="padding:8px 0 6px; font-weight:bold; color:#e67e22">{total}</td>
    </tr>
  </table>
</div>
"""))

hindi_dict,84
punjabi_dict,67
learned_hindi,26
learned_punjabi,19
TOTAL,196


In [42]:
# ══════════════════════════════════════════════════════
# LIST ALL LEARNED WORDS
# ══════════════════════════════════════════════════════

learned = model.list_learned()

for lang, words in learned.items():
    print(f"\n📖 Learned words [{lang}] — {len(words)} entries")
    print("─" * 40)
    if not words:
        print("  (none yet)")
    else:
        for native, roman in sorted(words.items()):
            print(f"  {native:<20} ↔  {roman}")


📖 Learned words [hindi] — 26 entries
────────────────────────────────────────
  आपका स्वागत          ↔  aapka swagat
  गाँव                 ↔  gaon
  चाय                  ↔  chai
  जंगल                 ↔  jangal
  ज़िंदगी              ↔  zindagi
  जुगाड़               ↔  jugaad
  दाल                  ↔  daal
  दुनिया               ↔  duniya
  देश                  ↔  desh
  नदी                  ↔  nadi
  पहाड़                ↔  pahaad
  पापा                 ↔  papa
  बहन                  ↔  behen
  बाजार                ↔  bazaar
  बारिश                ↔  baarish
  बॉलीवुड              ↔  bollywood
  भाई                  ↔  bhai
  मम्मी                ↔  mammi
  मिठाई                ↔  mithai
  मेरा नाम             ↔  mera naam
  रिक्शा               ↔  riksha
  रोटी                 ↔  roti
  शहर                  ↔  shahar
  सब्जी                ↔  sabzi
  समोसा                ↔  samosa
  हवा                  ↔  hawa

📖 Learned words [punjabi] — 19 entries
──────────────────────────────

In [43]:
# ══════════════════════════════════════════════════════
# FORGET A WORD (optional)
# ══════════════════════════════════════════════════════

# ✏️ Edit to remove a word:
# print(model.forget('रिक्शा', 'hindi'))
# print(model.forget('ਲੰਗਰ', 'punjabi'))

print("(Uncomment lines above to remove specific learned words)")

(Uncomment lines above to remove specific learned words)


---
## 10. Fuzzy Matching Demo
The model handles common typos and alternate spellings.

In [44]:
# ══════════════════════════════════════════════════════
# FUZZY MATCHING — typo tolerance
# ══════════════════════════════════════════════════════

fuzzy_tests = [
    # Typos in romanized Hindi
    ('namasthe', 'hindi'),     # typo of 'namaste'
    ('dhanyavad', 'hindi'),    # short form of 'dhanyavaad'
    ('khaana', 'hindi'),       # close to 'khaana'
    ('theeck', 'hindi'),       # typo of 'theek'
    # Typos in romanized Punjabi
    ('gher', 'punjabi'),       # close to 'ghar'
    ('paanee', 'punjabi'),     # alternate spelling of 'paani'
    ('dhanvad', 'punjabi'),    # short form of 'dhanvaad'
]

print("🔍 Fuzzy Matching — Typo Handling")
print("─" * 55)
for roman_typo, lang in fuzzy_tests:
    result = model.deromanize(roman_typo, target_lang=lang)
    print(f"  '{roman_typo}'  [{lang}]  →  {result}")

🔍 Fuzzy Matching — Typo Handling
───────────────────────────────────────────────────────
  'namasthe'  [hindi]  →  नमस्ते
  'dhanyavad'  [hindi]  →  धन्यवाद
  'khaana'  [hindi]  →  खाना
  'theeck'  [hindi]  →  ठीक
  'gher'  [punjabi]  →  ਘਰ
  'paanee'  [punjabi]  →  ਪਆਨਈ
  'dhanvad'  [punjabi]  →  ਧੰਨਵਾਦ


---
## 11. Quick Reference

```python
# Create model
model = RomanizationModel()

# Romanize native → roman
model.romanize('नमस्ते', lang='hindi')      # 'namaste'
model.romanize('ਘਰ', lang='punjabi')         # 'ghar'

# De-romanize roman → native
model.deromanize('namaste', 'hindi')          # 'नमस्ते'
model.deromanize('ghar', 'punjabi')           # 'ਘਰ'

# Auto-detect
model.process('नमस्ते')                       # auto hindi→roman
model.process('namaste', target_lang='hindi') # auto roman→hindi

# Learn new words
model.learn('रिक्शा', 'riksha', 'hindi')
model.learn('ਭੰਗੜਾ', 'bhangra', 'punjabi')

# Train from CSV
train_from_csv('train.csv', model)

# Inspect
model.vocab_size()
model.list_learned('hindi')
model.forget('रिक्शा', 'hindi')
```

---
*Romanization Language Model — Pure Python, no external dependencies.*